### Data Loading

We'll be working with user ratings, anime details, and user details.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
user_filtered = pd.read_csv('/content/drive/My Drive/Colab Notebooks/user-filtered.csv.zip')
anime_filtered = pd.read_csv('/content/drive/My Drive/Colab Notebooks/anime-filtered.csv.zip')
users_details = pd.read_csv('/content/drive/My Drive/Colab Notebooks/users-details-2023.csv.zip')
reviews = pd.read_csv('/content/drive/My Drive/Colab Notebooks/jikan_crawled_reviews_with_userid.csv')

print(f"User filtered shape: {user_filtered.shape}")
print(f"Anime filtered shape: {anime_filtered.shape}")
print(f"Users details shape: {users_details.shape}")
print(f"Reviews shape: {reviews.shape}")

Mounted at /content/drive
User filtered shape: (109224747, 3)
Anime filtered shape: (14952, 25)
Users details shape: (731290, 16)
Reviews shape: (19870, 13)


###Package Import

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

### Two tower temporal model



In [ ]:
anime_clean = anime_filtered.copy()

# Standardize column names
anime_clean.columns = [col.strip().replace(' ', '_') for col in anime_clean.columns]
print(f"Original columns: {anime_clean.columns.tolist()}")
if 'sypnopsis' in anime_clean.columns:
    anime_clean = anime_clean.rename(columns={'sypnopsis': 'Synopsis'})

# Handle missing values
anime_clean['Score'] = pd.to_numeric(anime_clean['Score'], errors='coerce').fillna(0)
anime_clean['Episodes'] = anime_clean['Episodes'].replace('Unknown', 0)
anime_clean['Episodes'] = pd.to_numeric(anime_clean['Episodes'], errors='coerce').fillna(0)
anime_clean['Genres'] = anime_clean['Genres'].fillna('Unknown')
anime_clean['Type'] = anime_clean['Type'].fillna('Unknown')
if 'Synopsis' in anime_clean.columns:
    anime_clean['Synopsis'] = anime_clean['Synopsis'].fillna('')

# Remove duplicates
anime_clean = anime_clean.drop_duplicates(subset=['anime_id']).reset_index(drop=True)
print(f"Cleaned anime: {len(anime_clean):,} unique anime")

# 1.2 Clean Users Details Dataset
users_clean = users_details.copy()
users_clean.columns = [col.strip().replace(' ', '_') for col in users_clean.columns]

# Convert date columns
date_cols = ['Birthday', 'Joined']
for col in date_cols:
    if col in users_clean.columns:
        users_clean[col] = pd.to_datetime(users_clean[col], errors='coerce')

# Fill numeric columns
numeric_cols = ['Days_Watched', 'Mean_Score', 'Episodes_Watched']
for col in numeric_cols:
    if col in users_clean.columns:
        users_clean[col] = pd.to_numeric(users_clean[col], errors='coerce').fillna(0)

print(f"Cleaned users: {len(users_clean):,} users")

# 1.3 Clean Ratings Dataset
ratings_clean = user_filtered.copy()
ratings_clean.columns = ['user_id', 'anime_id', 'rating']

# Check for -1 ratings (watched but not rated)
print(f"Original ratings: {len(ratings_clean):,}")
neg_ratings = (ratings_clean['rating'] == -1).sum()
print(f"Ratings with -1: {neg_ratings:,} ({neg_ratings/len(ratings_clean)*100:.2f}%)")

# DECISION: Remove -1 ratings for cleaner explicit feedback
ratings_clean = ratings_clean[ratings_clean['rating'] != -1].copy()
print(f"After removing -1: {len(ratings_clean):,} ratings")

# 1.4 Clean Reviews Dataset
reviews_clean = reviews.copy()
reviews_clean.columns = [col.strip().replace(' ', '_') for col in reviews_clean.columns]

print(f"Original reviews: {len(reviews_clean):,}")
print("Review columns:", reviews_clean.columns.tolist())

# Convert key columns
reviews_clean['review_date_utc'] = pd.to_datetime(reviews_clean['review_date_utc'], utc=True, errors='coerce')
reviews_clean['review_score'] = pd.to_numeric(reviews_clean['review_score'], errors='coerce')

# Keep only needed columns if they exist
needed_cols = ['user_id', 'anime_id', 'review_date_utc', 'review_score', 'review_text']
existing_needed_cols = [c for c in needed_cols if c in reviews_clean.columns]
reviews_clean = reviews_clean[existing_needed_cols].copy()

# Drop missing critical fields
reviews_clean = reviews_clean.dropna(subset=['user_id', 'anime_id', 'review_date_utc', 'review_score']).copy()

# Remove duplicates
reviews_clean = reviews_clean.drop_duplicates(subset=['user_id', 'anime_id', 'review_date_utc']).reset_index(drop=True)

print(f"Cleaned reviews: {len(reviews_clean):,}")

data = reviews_clean.merge(
    anime_clean[['anime_id', 'Name', 'Genres', 'Type', 'Episodes', 'Score', 'Synopsis']]
    if 'Synopsis' in anime_clean.columns
    else anime_clean[['anime_id', 'Name', 'Genres', 'Type', 'Episodes', 'Score']],
    on='anime_id',
    how='left'
)

data = data.dropna(subset=['user_id', 'anime_id']).copy()
data = data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print(f"Merged review-anime rows: {len(data):,}")

# Keep review text as string
data['review_text'] = data['review_text'].fillna('').astype(str)

print(f"Merged review-anime rows: {len(data):,}")
print(data[['user_id', 'anime_id', 'review_date_utc', 'review_text']].head())

# Keep only users with enough interaction history
user_counts = data.groupby('user_id').size()
valid_users = user_counts[user_counts >= 5].index

data = data[data['user_id'].isin(valid_users)].copy()
data = data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print(f"Users kept: {data['user_id'].nunique():,}")
print(f"Rows kept: {len(data):,}")

# Binary label for retrieval task
data['label'] = (data['review_score'] >= 7).astype(int)

print("Label distribution:")
print(data['label'].value_counts(dropna=False))
print(data['label'].value_counts(normalize=True, dropna=False))


Original columns: ['anime_id', 'Name', 'Score', 'Genres', 'English_name', 'Japanese_name', 'sypnopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped']
Cleaned anime: 14,952 unique anime
Cleaned users: 731,290 users
Original ratings: 109,224,747
Ratings with -1: 0 (0.00%)
After removing -1: 109,224,747 ratings
Original reviews: 19,870
Review columns: ['user_id', 'username', 'review_id', 'anime_id', 'entry_title', 'review_date_utc', 'review_text', 'entry_url', 'review_url', 'review_score', 'is_spoiler', 'is_preliminary', 'crawl_timestamp_utc']
Cleaned reviews: 19,870
Merged review-anime rows: 19,870
Merged review-anime rows: 19,870
   user_id  anime_id           review_date_utc  \
0        1         1 2006-11-07 18:34:00+00:00   
1        1       263 2006-11-08 14:39:00+00:00   
2        1       565 2006-12-24 09:46:00+00:00   
3  

In [ ]:
# Ensure that for each users, earlier interactions are in train, later interactions in val/test
def chrono_split_user(df, train_frac=0.8, val_frac=0.1):
    df = df.sort_values('review_date_utc').copy()
    n = len(df)

    train_end = max(1, int(n * train_frac))
    val_end = max(train_end + 1, int(n * (train_frac + val_frac))) if n >= 3 else n

    df['split'] = 'test'
    df.iloc[:train_end, df.columns.get_loc('split')] = 'train'
    df.iloc[train_end:val_end, df.columns.get_loc('split')] = 'val'
    return df

data = (
    data.groupby('user_id', group_keys=False)
    .apply(chrono_split_user)
    .reset_index(drop=True)
)

print(data['split'].value_counts())
print(data['split'].value_counts(normalize=True))


split
train    9419
test     1375
val      1364
Name: count, dtype: int64
split
train    0.774716
test     0.113094
val      0.112190
Name: proportion, dtype: float64


/tmp/ipykernel_4368/835617764.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(chrono_split_user)


In [ ]:
# Build leakage-safe past review text history
data = data.sort_values(['user_id', 'review_date_utc']).copy()

# Number of past interactions before the current one
data['hist_count'] = data.groupby('user_id').cumcount()

# Past score features only
data['hist_mean_score'] = (
    data.groupby('user_id')['review_score']
    .transform(lambda s: s.expanding().mean().shift(1))
)

data['hist_std_score'] = (
    data.groupby('user_id')['review_score']
    .transform(lambda s: s.expanding().std().shift(1))
)

data['hist_mean_score'] = data['hist_mean_score'].fillna(0)
data['hist_std_score'] = data['hist_std_score'].fillna(0)

def build_past_review_text(df):
    df = df.sort_values('review_date_utc').copy()
    history = []
    past_texts = []

    for txt in df['review_text'].fillna('').astype(str):
        past_texts.append(' '.join(history))   # only past reviews
        history.append(txt)                    # add current review after

    df['past_review_text'] = past_texts
    return df

data = (
    data.groupby('user_id', group_keys=False)
    .apply(build_past_review_text)
    .reset_index(drop=True)
)

print(
    data[['user_id', 'anime_id', 'review_score', 'hist_count',
          'hist_mean_score', 'hist_std_score', 'past_review_text', 'label', 'split']].head(10)
)

# Build historical genre preference features
data['Genres'] = data['Genres'].fillna('')

data['genre_list'] = data['Genres'].apply(
    lambda x: [g.strip() for g in str(x).split(',') if g.strip() != '']
)

all_genres = sorted(set(g for genres in data['genre_list'] for g in genres))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}

print(f"Number of genres: {len(all_genres)}")
print(all_genres[:10])

   user_id  anime_id  review_score  hist_count  hist_mean_score  \
0        1         1            10           0         0.000000   
1        1       263            10           1        10.000000   
2        1       565             8           2        10.000000   
3        1       853             8           3         9.333333   
4        1       849            10           4         9.000000   
5        1         7             8           5         9.200000   
6        1         6            10           6         9.000000   
7        1        25             8           7         9.142857   
8        1      1519             6           8         9.000000   
9        1      1210             8           9         8.666667   

   hist_std_score                                   past_review_text  label  \
0        0.000000                                                         1   
1        0.000000  Cowboy Bebop is an episodic series. By episodi...      1   
2        0.000000  Cowboy

/tmp/ipykernel_4368/2753792287.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_past_review_text)


In [ ]:
# 2.3 Create past-only user genre history vectors

def build_past_genre_matrix(df, genre_to_idx):
    df = df.sort_values('review_date_utc').copy()
    num_genres = len(genre_to_idx)

    running = np.zeros(num_genres, dtype=np.float32)
    hist_vectors = []

    for _, row in df.iterrows():
        hist_vectors.append(running.copy())

        for g in row['genre_list']:
            if g in genre_to_idx:
                running[genre_to_idx[g]] += 1.0

    cols = [f'user_genre_{g}' for g in genre_to_idx.keys()]
    hist_df = pd.DataFrame(hist_vectors, columns=cols, index=df.index)
    return hist_df

genre_feature_parts = []

for user_id, df_user in data.groupby('user_id'):
    hist_df = build_past_genre_matrix(df_user, genre_to_idx)
    genre_feature_parts.append(hist_df)

genre_feature_df = pd.concat(genre_feature_parts).sort_index()

genre_cols = genre_feature_df.columns.tolist()

# Drop old genre-history columns if they already exist from a previous run
old_genre_cols = [c for c in data.columns if c.startswith('user_genre_')]
if old_genre_cols:
    data = data.drop(columns=old_genre_cols)

# join back to data
data = data.join(genre_feature_df)

row_sums = data[genre_cols].sum(axis=1)
row_sums = row_sums.replace(0, 1)

data[genre_cols] = data[genre_cols].div(row_sums, axis=0)

print(data[['user_id', 'anime_id', 'genre_list'] + genre_cols[:5]].head())
print(data[genre_cols].shape)


   user_id  anime_id                                         genre_list  \
0        1         1  [Action, Adventure, Comedy, Drama, Sci-Fi, Space]   
1        1       263                   [Comedy, Sports, Drama, Shounen]   
2        1       565  [Action, Military, Sci-Fi, Adventure, Historic...   
3        1       853           [Comedy, Harem, Romance, School, Shoujo]   
4        1       849  [Comedy, Mystery, Parody, School, Sci-Fi, Slic...   

   user_genre_Action  user_genre_Adventure  user_genre_Cars  \
0           0.000000              0.000000              0.0   
1           0.166667              0.166667              0.0   
2           0.100000              0.100000              0.0   
3           0.125000              0.125000              0.0   
4           0.095238              0.095238              0.0   

   user_genre_Comedy  user_genre_Dementia  
0           0.000000                  0.0  
1           0.166667                  0.0  
2           0.200000                  

In [ ]:
# Create one unique row per anime
item_df = data[['anime_id', 'Synopsis', 'genre_list', 'Type', 'Episodes', 'Name']].drop_duplicates('anime_id').copy()

# Fill any remaining missing values
item_df['Synopsis'] = item_df['Synopsis'].fillna('')
item_df['Type'] = item_df['Type'].fillna('Unknown')
item_df['Episodes'] = pd.to_numeric(item_df['Episodes'], errors='coerce').fillna(0)

# TF-IDF on synopsis
tfidf = TfidfVectorizer(max_features= 100, stop_words='english')
item_tfidf = tfidf.fit_transform(item_df['Synopsis']).astype(np.float32)

# Multi-hot encode genres
mlb = MultiLabelBinarizer()
item_genres = mlb.fit_transform(item_df['genre_list'])

# One-hot encode anime type
type_dummies = pd.get_dummies(item_df['Type'], prefix='type')

# Scale episodes
scaler = StandardScaler()
item_episodes = scaler.fit_transform(item_df[['Episodes']])

# Combine dense item features
item_dense = np.hstack([
    item_genres.astype(np.float32),
    type_dummies.values.astype(np.float32),
    item_episodes.astype(np.float32)
])

# Mapping from anime_id to item row
anime_id_to_row = dict(zip(item_df['anime_id'].values, np.arange(len(item_df))))
row_to_anime_id = dict(zip(np.arange(len(item_df)), item_df['anime_id'].values))

print("item_df shape:", item_df.shape)
print("item_tfidf shape:", item_tfidf.shape)
print("item_dense shape:", item_dense.shape)

item_df shape: (5078, 6)
item_tfidf shape: (5078, 100)
item_dense shape: (5078, 51)


In [ ]:
# 2.4b Add safe past-only user history feature: mean episodes watched

data['item_episodes'] = pd.to_numeric(data['Episodes'], errors='coerce').fillna(0)

data['hist_mean_episodes'] = (
    data.groupby('user_id')['item_episodes']
    .transform(lambda s: s.expanding().mean().shift(1))
    .fillna(0)
)

print(data[['user_id', 'anime_id', 'item_episodes', 'hist_mean_episodes']].head(10))

   user_id  anime_id  item_episodes  hist_mean_episodes
0        1         1           26.0            0.000000
1        1       263           75.0           26.000000
2        1       565            1.0           50.500000
3        1       853           26.0           34.000000
4        1       849           14.0           32.000000
5        1         7           26.0           28.400000
6        1         6           26.0           28.000000
7        1        25           24.0           27.714286
8        1      1519           12.0           27.250000
9        1      1210           24.0           25.555556


In [ ]:
# 2.5 Keep only rows usable for the two-tower model
model_data = data[data['hist_count'] > 0].copy()
model_data = model_data[model_data['anime_id'].isin(anime_id_to_row)].copy()

model_data = model_data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print("model_data shape:", model_data.shape)
print(model_data['split'].value_counts())
print("Unique users:", model_data['user_id'].nunique())
print("Unique anime:", model_data['anime_id'].nunique())

# Build TF-IDF features from PAST review text only
train_mask = model_data['split'] == 'train'

user_text_vectorizer = TfidfVectorizer(
    max_features=300,
    stop_words='english'
)

user_text_train = user_text_vectorizer.fit_transform(
    model_data.loc[train_mask, 'past_review_text']
).astype(np.float32)

user_text_other = user_text_vectorizer.transform(
    model_data.loc[~train_mask, 'past_review_text']
).astype(np.float32)

# Create full matrix aligned to model_data row order
from scipy.sparse import vstack

user_text_features = np.zeros((len(model_data), len(user_text_vectorizer.get_feature_names_out())), dtype=np.float32)
user_text_features[train_mask.values] = user_text_train.toarray()
user_text_features[~train_mask.values] = user_text_other.toarray()

print("user_text_features shape:", user_text_features.shape)

# Encode users into 0..N-1
user_encoder = LabelEncoder()
model_data['user_idx'] = user_encoder.fit_transform(model_data['user_id'])

# Map anime_id to item row index
model_data['item_idx'] = model_data['anime_id'].map(anime_id_to_row)

# Drop any rows that somehow failed item mapping
model_data = model_data.dropna(subset=['item_idx']).copy()
model_data['item_idx'] = model_data['item_idx'].astype(int)

num_users = model_data['user_idx'].nunique()
num_items = len(item_df)



print("model_data shape:", model_data.shape)
print("num_users:", num_users)
print("num_items:", num_items)

model_data[['user_id', 'user_idx', 'anime_id', 'item_idx', 'split']].head()

model_data shape: (11336, 63)
split
train    8597
test     1375
val      1364
Name: count, dtype: int64
Unique users: 822
Unique anime: 4950
user_text_features shape: (11336, 300)
model_data shape: (11336, 65)
num_users: 822
num_items: 5078


,user_id,user_idx,anime_id,item_idx,split
0,1,0,263,1,train
1,1,0,565,2,train
2,1,0,853,3,train
3,1,0,849,4,train
4,1,0,7,5,train


In [ ]:
# 2.7 Create positive and negative pairs for two-tower training
# positive pair = actual user interaction
# negative pair = anime the user had not seen before that time

rng = np.random.default_rng(42)

all_item_ids = set(item_df['anime_id'].tolist())
model_data = model_data.sort_values(['user_id', 'review_date_utc']).copy()

user_seen_so_far = {}
rows = []

for idx, row in model_data.iterrows():
    u = row['user_id']
    i = row['anime_id']

    if u not in user_seen_so_far:
        user_seen_so_far[u] = set()

    seen_before = user_seen_so_far[u].copy()

    # only rows with review_score >= 7 become positive examples
    if row['label'] == 1 and len(seen_before) > 0:
        rows.append({
            'user_idx': row['user_idx'],
            'item_idx': row['item_idx'],
            'label': 1,
            'split': row['split'],
            'row_id': idx
        })

        candidate_neg = list(all_item_ids - seen_before - {i})

        if len(candidate_neg) > 0:
            neg_items = rng.choice(
                candidate_neg,
                size=min(9, len(candidate_neg)),
                replace=False
            )

            for neg_anime in neg_items:
                rows.append({
                    'user_idx': row['user_idx'],
                    'item_idx': anime_id_to_row[neg_anime],
                    'label': 0,
                    'split': row['split'],
                    'row_id': idx
                })

    # update history only after current row
    user_seen_so_far[u].add(i)

pair_df = pd.DataFrame(rows)

print("pair_df shape:", pair_df.shape)
print(pair_df['label'].value_counts())
print(pair_df['split'].value_counts())
pair_df.head()

pair_df shape: (75450, 5)
label
0    67905
1     7545
Name: count, dtype: int64
split
train    55940
test      9810
val       9700
Name: count, dtype: int64


,user_idx,item_idx,label,split,row_id
0,0,2,1,train,1
1,0,627,0,train,1
2,0,4110,0,train,1
3,0,4350,0,train,1
4,0,4857,0,train,1


In [ ]:
# 2.8 Attach leakage-safe user text history features to pair_df
# and also merge them back into model_data for recommendation use

# Reset clean row_id aligned with user_text_features
model_data = model_data.reset_index(drop=True).copy()
model_data['row_id'] = model_data.index

# Remove any old user_text_* columns first to avoid _x / _y duplication
old_model_text_cols = [c for c in model_data.columns if c.startswith('user_text_')]
if old_model_text_cols:
    model_data = model_data.drop(columns=old_model_text_cols)

old_pair_text_cols = [c for c in pair_df.columns if c.startswith('user_text_')]
if old_pair_text_cols:
    pair_df = pair_df.drop(columns=old_pair_text_cols)

# Build one text-feature row per original interaction row
row_feature_df = pd.DataFrame(user_text_features)
row_feature_df.columns = [f'user_text_{i}' for i in range(row_feature_df.shape[1])]
row_feature_df['row_id'] = model_data['row_id'].values

# Merge text features into model_data
model_data = model_data.merge(row_feature_df, on='row_id', how='left')

# Columns to bring from model_data into pair_df
base_hist_cols = ['row_id', 'hist_count', 'hist_mean_score', 'hist_std_score']

# include optional columns if they exist
optional_hist_cols = [c for c in ['hist_mean_episodes'] if c in model_data.columns]
genre_hist_cols = [c for c in model_data.columns if c.startswith('genre_hist_')]
user_text_cols = [c for c in model_data.columns if c.startswith('user_text_')]

cols_to_merge = base_hist_cols + optional_hist_cols + genre_hist_cols + user_text_cols

# Remove any already-existing duplicate cols in pair_df except row_id
dup_cols = [c for c in cols_to_merge if c in pair_df.columns and c != 'row_id']
if dup_cols:
    pair_df = pair_df.drop(columns=dup_cols)

# Merge all user-side features into pair_df
pair_df = pair_df.merge(
    model_data[cols_to_merge],
    on='row_id',
    how='left'
)

print("pair_df shape after merge:", pair_df.shape)
print("model_data shape after merge:", model_data.shape)
print("Number of user text features:", len(user_text_cols))
print("Missing text features in pair_df:", pair_df[user_text_cols].isna().sum().sum())
print("Missing text features in model_data:", model_data[user_text_cols].isna().sum().sum())

display_cols = ['row_id', 'user_idx', 'item_idx', 'label', 'split'] + user_text_cols[:5]
print(pair_df[display_cols].head())

pair_df shape after merge: (75450, 309)
model_data shape after merge: (11336, 366)
Number of user text features: 300
Missing text features in pair_df: 0
Missing text features in model_data: 0
   row_id  user_idx  item_idx  label  split  user_text_0  user_text_1  \
0       1         0         2      1  train          0.0     0.092577   
1       1         0       627      0  train          0.0     0.092577   
2       1         0      4110      0  train          0.0     0.092577   
3       1         0      4350      0  train          0.0     0.092577   
4       1         0      4857      0  train          0.0     0.092577   

   user_text_2  user_text_3  user_text_4  
0          0.0          0.0          0.0  
1          0.0          0.0          0.0  
2          0.0          0.0          0.0  
3          0.0          0.0          0.0  
4          0.0          0.0          0.0  


In [ ]:
# 2.9 Split pair_df into train / val / test

train_df = pair_df[pair_df['split'] == 'train'].reset_index(drop=True)
val_df   = pair_df[pair_df['split'] == 'val'].reset_index(drop=True)
test_df  = pair_df[pair_df['split'] == 'test'].reset_index(drop=True)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print(test_df['label'].value_counts())

train: (55940, 309)
val: (9700, 309)
test: (9810, 309)
label
0    50346
1     5594
Name: count, dtype: int64
label
0    8730
1     970
Name: count, dtype: int64
label
0    8829
1     981
Name: count, dtype: int64


In [ ]:
# 3.0 Build PyTorch datasets and dataloaders

import torch
from torch.utils.data import Dataset, DataLoader

base_user_hist_cols = ['hist_count', 'hist_mean_score', 'hist_std_score']

# include optional numeric history features if they exist
optional_cols = []
for c in ['hist_mean_episodes']:
    if c in pair_df.columns:
        optional_cols.append(c)

# include genre history if it exists
genre_cols_existing = [c for c in pair_df.columns if c.startswith('genre_hist_')]

# final combined feature set
user_hist_cols = base_user_hist_cols + optional_cols + genre_cols_existing + user_text_cols

print("Total user feature columns:", len(user_hist_cols))
print(user_hist_cols[:10])

class TwoTowerDataset(Dataset):
    def __init__(self, df, user_hist_cols):
        self.df = df.reset_index(drop=True)

        self.user_idx = self.df['user_idx'].values.astype(np.int64)
        self.item_idx = self.df['item_idx'].values.astype(np.int64)
        self.user_hist = self.df[user_hist_cols].values.astype(np.float32)
        self.labels = self.df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'user_idx': torch.tensor(self.user_idx[idx], dtype=torch.long),
            'item_idx': torch.tensor(self.item_idx[idx], dtype=torch.long),
            'user_hist': torch.tensor(self.user_hist[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_dataset = TwoTowerDataset(train_df, user_hist_cols)
val_dataset   = TwoTowerDataset(val_df, user_hist_cols)
test_dataset  = TwoTowerDataset(test_df, user_hist_cols)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Total user feature columns: 304
['hist_count', 'hist_mean_score', 'hist_std_score', 'hist_mean_episodes', 'user_text_0', 'user_text_1', 'user_text_2', 'user_text_3', 'user_text_4', 'user_text_5']
Train batches: 219
Val batches: 38
Test batches: 39


In [ ]:
# 3.1 Convert item features to tensors

item_tfidf_dense = item_tfidf.toarray().astype(np.float32)
item_dense = item_dense.astype(np.float32)

item_tfidf_tensor = torch.tensor(item_tfidf_dense, dtype=torch.float32)
item_dense_tensor = torch.tensor(item_dense, dtype=torch.float32)

print("item_tfidf_tensor:", item_tfidf_tensor.shape)
print("item_dense_tensor:", item_dense_tensor.shape)

item_tfidf_tensor: torch.Size([5078, 100])
item_dense_tensor: torch.Size([5078, 51])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, user_hist_dim, item_tfidf_dim, item_dense_dim, emb_dim=32):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, 64)

        self.user_mlp = nn.Sequential(
            nn.Linear(64 + user_hist_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, emb_dim)
        )

        self.item_mlp = nn.Sequential(
            nn.Linear(item_tfidf_dim + item_dense_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, emb_dim)
        )

    def forward(self, user_idx, user_hist, item_idx, item_tfidf_tensor, item_dense_tensor):
        user_emb = self.user_embedding(user_idx)
        user_input = torch.cat([user_emb, user_hist], dim=1)
        user_vec = self.user_mlp(user_input)

        item_input = torch.cat([
            item_tfidf_tensor[item_idx],
            item_dense_tensor[item_idx]
        ], dim=1)
        item_vec = self.item_mlp(item_input)

        score = 3.2 * (F.normalize(user_vec, dim=1) * F.normalize(item_vec, dim=1)).sum(dim=1)
        return score

In [ ]:
# 3.3 Initialize model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TwoTowerModel(
    num_users=num_users,
    user_hist_dim=len(user_hist_cols),
    item_tfidf_dim=item_tfidf_tensor.shape[1],
    item_dense_dim=item_dense_tensor.shape[1],
    emb_dim=64
).to(device)

item_tfidf_tensor = item_tfidf_tensor.to(device)
item_dense_tensor = item_dense_tensor.to(device)

print(model)
print("Device:", device)

TwoTowerModel(
  (user_embedding): Embedding(822, 64)
  (user_mlp): Sequential(
    (0): Linear(in_features=368, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
  )
  (item_mlp): Sequential(
    (0): Linear(in_features=151, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
  )
)
Device: cuda


In [ ]:
# 3.4 Train the two-tower model

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_loss = float('inf')
patience = 5
patience_counter = 0
num_epochs = 20

for epoch in range(num_epochs):
    # -------- train --------
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= max(1, len(train_loader))

    # -------- validation --------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            user_idx = batch['user_idx'].to(device)
            item_idx = batch['item_idx'].to(device)
            user_hist = batch['user_hist'].to(device)
            labels = batch['label'].to(device)

            logits = model(
                user_idx=user_idx,
                user_hist=user_hist,
                item_idx=item_idx,
                item_tfidf_tensor=item_tfidf_tensor,
                item_dense_tensor=item_dense_tensor
            )

            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= max(1, len(val_loader))

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # -------- early stopping --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_two_tower.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1: train_loss=0.3176, val_loss=0.3096
Epoch 2: train_loss=0.3012, val_loss=0.3045
Epoch 3: train_loss=0.2944, val_loss=0.3010
Epoch 4: train_loss=0.2896, val_loss=0.3013
Epoch 5: train_loss=0.2856, val_loss=0.2977
Epoch 6: train_loss=0.2822, val_loss=0.2976
Epoch 7: train_loss=0.2793, val_loss=0.2973
Epoch 8: train_loss=0.2775, val_loss=0.2981
Epoch 9: train_loss=0.2754, val_loss=0.2949
Epoch 10: train_loss=0.2731, val_loss=0.2946
Epoch 11: train_loss=0.2714, val_loss=0.2945
Epoch 12: train_loss=0.2696, val_loss=0.2957
Epoch 13: train_loss=0.2688, val_loss=0.2952
Epoch 14: train_loss=0.2665, val_loss=0.2954
Epoch 15: train_loss=0.2652, val_loss=0.2934
Epoch 16: train_loss=0.2634, val_loss=0.2940
Epoch 17: train_loss=0.2629, val_loss=0.2953
Epoch 18: train_loss=0.2614, val_loss=0.2939
Epoch 19: train_loss=0.2609, val_loss=0.2934
Epoch 20: train_loss=0.2588, val_loss=0.2932


In [ ]:
# 3.5 Evaluate on test set

from sklearn.metrics import roc_auc_score, accuracy_score

model.load_state_dict(torch.load('best_two_tower.pt', map_location=device))
model.eval()

criterion = nn.BCEWithLogitsLoss()

all_logits = []
all_probs = []
all_labels = []
test_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in test_loader:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)
        labels = batch['label'].to(device)

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        loss = criterion(logits, labels)
        test_loss += loss.item()
        num_batches += 1

        probs = torch.sigmoid(logits)

        all_logits.extend(logits.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_logits = np.array(all_logits)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

preds = (all_probs >= 0.5).astype(int)
test_bce = test_loss / max(1, num_batches)

print("Test BCE:", round(test_bce, 4))
print("Test AUC:", round(roc_auc_score(all_labels, all_probs), 4))
print("Test Accuracy:", round(accuracy_score(all_labels, preds), 4))

Test BCE: 0.303
Test AUC: 0.6864
Test Accuracy: 0.9008


In [ ]:
# 3.6 Ranking metric helpers

import numpy as np
import pandas as pd

def precision_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    return np.sum(relevances) / k

def hitrate_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    return 1.0 if np.sum(relevances) > 0 else 0.0

def mrr_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    for idx, rel in enumerate(relevances, start=1):
        if rel == 1:
            return 1.0 / idx
    return 0.0

def ap_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    num_relevant = np.sum(relevances)
    if num_relevant == 0:
        return 0.0

    precisions = []
    hits = 0
    for i, rel in enumerate(relevances, start=1):
        if rel == 1:
            hits += 1
            precisions.append(hits / i)

    return np.mean(precisions) if precisions else 0.0

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    return np.sum((2**relevances - 1) / np.log2(np.arange(2, len(relevances) + 2)))

def ndcg_at_k(relevances, k):
    actual_dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevances, reverse=True)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

def recall_at_k(relevances, k):
    relevances = np.asarray(relevances)
    total_relevant = np.sum(relevances)
    if total_relevant == 0:
        return 0.0
    return np.sum(relevances[:k]) / total_relevant

In [ ]:
# 3.7 Score every row in test_df for ranking evaluation

model.load_state_dict(torch.load('best_two_tower.pt', map_location=device))
model.eval()

test_eval_df = test_df.copy().reset_index(drop=True)

test_dataset_eval = TwoTowerDataset(test_eval_df, user_hist_cols)
test_loader_eval = DataLoader(test_dataset_eval, batch_size=256, shuffle=False)

all_logits = []

with torch.no_grad():
    for batch in test_loader_eval:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        all_logits.extend(logits.cpu().numpy())

test_eval_df['score'] = np.array(all_logits)

print(test_eval_df[['user_idx', 'item_idx', 'label', 'score']].head())
print(test_eval_df.shape)

   user_idx  item_idx  label     score
0         0        15      1 -0.849241
1         0      4917      0 -2.194836
2         0       546      0 -2.949508
3         0      1443      0 -2.404528
4         0      3726      0 -2.004952
(9810, 310)


In [ ]:
# 3.8 Check ranking groups

print("Unique ranking groups in test:", test_eval_df['row_id'].nunique())
print(test_eval_df.groupby('row_id').size().value_counts().sort_index())

Unique ranking groups in test: 981
10    981
Name: count, dtype: int64


In [ ]:
# 3.9 Compute sampled Recall@K and NDCG@K on test set

def evaluate_ranking_metrics(
    df,
    group_col='row_id',
    label_col='label',
    score_col='score',
    ks=[1, 2, 3, 5 ,10]
):
    results = {}
    grouped = df.groupby(group_col)

    for k in ks:
        precisions = []
        recalls = []
        hitrates = []
        mrrs = []
        maps = []
        ndcgs = []

        for _, group in grouped:
            group_sorted = group.sort_values(score_col, ascending=False)
            relevances = group_sorted[label_col].tolist()

            precisions.append(precision_at_k(relevances, k))
            recalls.append(recall_at_k(relevances, k))
            hitrates.append(hitrate_at_k(relevances, k))
            mrrs.append(mrr_at_k(relevances, k))
            maps.append(ap_at_k(relevances, k))
            ndcgs.append(ndcg_at_k(relevances, k))

        results[f'Precision@{k}'] = np.mean(precisions)
        results[f'Recall@{k}'] = np.mean(recalls)
        results[f'HitRate@{k}'] = np.mean(hitrates)
        results[f'MRR@{k}'] = np.mean(mrrs)
        results[f'MAP@{k}'] = np.mean(maps)
        results[f'NDCG@{k}'] = np.mean(ndcgs)

    return results

ranking_results = evaluate_ranking_metrics(
    test_eval_df,
    group_col='row_id',
    label_col='label',
    score_col='score',
    ks=[1, 2, 3,5,10]
)

print("Ranking metrics on sampled test groups:")
for metric, value in ranking_results.items():
    print(f"{metric}: {value:.4f}")

Ranking metrics on sampled test groups:
Precision@1: 0.3150
Recall@1: 0.3150
HitRate@1: 0.3150
MRR@1: 0.3150
MAP@1: 0.3150
NDCG@1: 0.3150
Precision@2: 0.2441
Recall@2: 0.4883
HitRate@2: 0.4883
MRR@2: 0.4016
MAP@2: 0.4016
NDCG@2: 0.4243
Precision@3: 0.1916
Recall@3: 0.5749
HitRate@3: 0.5749
MRR@3: 0.4305
MAP@3: 0.4305
NDCG@3: 0.4676
Precision@5: 0.1519
Recall@5: 0.7594
HitRate@5: 0.7594
MRR@5: 0.4733
MAP@5: 0.4733
NDCG@5: 0.5442
Precision@10: 0.1000
Recall@10: 1.0000
HitRate@10: 1.0000
MRR@10: 0.5056
MAP@10: 0.5056
NDCG@10: 0.6222


In [ ]:
# 3.10 Recommend Top-N titles for one user

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

def recommend_top_n_for_user(
    user_id,
    model,
    model_data,
    item_df,
    item_tfidf_tensor,
    item_dense_tensor,
    user_hist_cols,
    top_n=10,
    exclude_seen=True,
    device='cpu'
):
    model.eval()

    # Get this user's rows in chronological order
    user_rows = model_data[model_data['user_id'] == user_id].sort_values('review_date_utc')

    if len(user_rows) == 0:
        raise ValueError(f"user_id {user_id} not found in model_data")

    # Use the latest available user state
    latest_row = user_rows.iloc[-1]

    user_idx = int(latest_row['user_idx'])
    user_hist = latest_row[user_hist_cols].to_numpy(dtype=np.float32)

    user_idx_tensor = torch.tensor([user_idx], dtype=torch.long, device=device)
    user_hist_tensor = torch.tensor(user_hist, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        # User vector
        user_emb = model.user_embedding(user_idx_tensor)
        user_input = torch.cat([user_emb, user_hist_tensor], dim=1)
        user_vec = model.user_mlp(user_input)
        user_vec = F.normalize(user_vec, dim=1)

        # Item vectors for all anime
        item_input = torch.cat([item_tfidf_tensor, item_dense_tensor], dim=1)
        item_vecs = model.item_mlp(item_input)
        item_vecs = F.normalize(item_vecs, dim=1)

        # Similarity scores
        scores = 3.2 * torch.matmul(item_vecs, user_vec.T).squeeze(1)

    rec_df = item_df[['anime_id', 'Name']].copy()
    rec_df['score'] = scores.cpu().numpy()

    if exclude_seen:
        seen_anime = set(user_rows['anime_id'].tolist())
        rec_df = rec_df[~rec_df['anime_id'].isin(seen_anime)].copy()

    rec_df = rec_df.sort_values('score', ascending=False).head(top_n).reset_index(drop=True)
    return rec_df


sample_user_id = model_data['user_id'].iloc[1503]

top10_recs = recommend_top_n_for_user(
    user_id=sample_user_id,
    model=model,
    model_data=model_data,
    item_df=item_df,
    item_tfidf_tensor=item_tfidf_tensor,
    item_dense_tensor=item_dense_tensor,
    user_hist_cols=user_hist_cols,
    top_n=10,
    exclude_seen=True,
    device=device
)

print(f"Top 10 recommendations for user_id = {sample_user_id}")
print(top10_recs)

Top 10 recommendations for user_id = 14494
   anime_id                                            Name     score
0      2921                                 Ashita no Joe 2 -0.561410
1       377                                       eX-Driver -0.739157
2     13171                                    Wasurenagumo -0.925563
3      8740          One Piece Film: Strong World Episode 0 -0.976004
4      1162                            Maze☆Bakunetsu Jikuu -1.019096
5     33263  Kubikiri Cycle: Aoiro Savant to Zaregototsukai -1.031872
6     11767                                         Justeen -1.036651
7      3218                        Gensoumaden Saiyuuki OVA -1.138192
8      6838                      Dekobou no Jidousha Ryokou -1.140142
9     34984                       Koi wa Ameagari no You ni -1.142769
